In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import time
import os
from surprise import Dataset, Reader, SVD,KNNBasic
from surprise.model_selection import train_test_split,cross_validate
from surprise import accuracy,dump
from collections import defaultdict


In [ ]:
!pip install fastparquet

### Data Preprocessing 

In [ ]:
file_paths = [
    'combined_data_1.txt', 
    'combined_data_2.txt', 
    'combined_data_3.txt', 
    'combined_data_4.txt'
]

df_list = []

print("Starting data ingestion with Date included. This will take a few minutes...")

for file in file_paths:
    print(f"Processing {file}...")
    raw_data = []
    
    with open(file, 'r') as f:
        current_movie = None
        for line in f:
            line = line.strip()
            if line.endswith(':'):
                current_movie = int(line[:-1]) 
            else:
                user, rating, date_str = line.split(',')
                raw_data.append([current_movie, int(user), int(rating), date_str])

    temp_df = pd.DataFrame(raw_data, columns=['Movie_ID', 'User_ID', 'Rating', 'Date'])
    
    temp_df['Rating'] = temp_df['Rating'].astype(np.int8)
    temp_df['Movie_ID'] = temp_df['Movie_ID'].astype(np.int32)
    temp_df['User_ID'] = temp_df['User_ID'].astype(np.int32)
    temp_df['Date'] = pd.to_datetime(temp_df['Date'])
    
    df_list.append(temp_df)
    
    del raw_data
    gc.collect() 

print("Concatenating files into a single matrix...")
df = pd.concat(df_list, ignore_index=True)

del df_list
gc.collect()

print(f"Total rows loaded: {len(df):,}")

In [ ]:
df.head(5)

In [ ]:
min_movie_ratings = 10
filter_movies = df['Movie_ID'].value_counts() >= min_movie_ratings
filter_movies = filter_movies[filter_movies].index.tolist()

min_user_ratings = 10
filter_users = df['User_ID'].value_counts() >= min_user_ratings
filter_users = filter_users[filter_users].index.tolist()

# Create a pruned dataframe
df = df[df['Movie_ID'].isin(filter_movies) & df['User_ID'].isin(filter_users)]

print(f"Original Rows: {len(df):,}")

### Data saving 

In [ ]:
parquet_filename = 'netflix_full_processed'
print(f"Saving df to '{parquet_filename}'...")

df.to_parquet(parquet_filename, engine='fastparquet', compression='snappy')

file_size_bytes = os.path.getsize(parquet_filename)
print(f"Saved successfully! Disk space used: {file_size_bytes / (1024 * 1024):.2f} MB")

In [10]:
movies_df = pd.read_csv('movie_titles.csv', encoding='ISO-8859-1', 
                        header=None, names=['Movie_ID', 'Year', 'Title'], 
                        on_bad_lines='skip')
movies_df['Movie_ID'] = movies_df['Movie_ID'].astype(np.int32)

In [3]:
movies_df.head(5)

,Movie_ID,Year,Title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW


### Exploratory Data Analysis 

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12})

print(f"Total Ratings: {len(df):,}")
print(f"Unique Users: {df['User_ID'].nunique():,}")
print(f"Unique Movies: {df['Movie_ID'].nunique():,}")

#### User activity patterns

In [ ]:
user_stats = df.groupby('User_ID')['Rating'].agg(['count', 'mean'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Number of ratings per user (Log scale due to Long Tail)
sns.histplot(user_stats['count'], bins=50, kde=False, ax=axes[0], color='skyblue')
axes[0].set_yscale('log')
axes[0].set_title('Distribution of Number of Ratings per User (Log Scale)')
axes[0].set_xlabel('Number of Ratings Given')
axes[0].set_ylabel('Number of Users (Log Scale)')

# Plot 2: Average rating given by users
sns.histplot(user_stats['mean'], bins=50, kde=True, ax=axes[1], color='salmon')
axes[1].set_title('Distribution of Average Ratings Given by Users')
axes[1].set_xlabel('Average Rating')
axes[1].set_ylabel('Number of Users')

plt.tight_layout()
plt.show()

print("User Activity Summary:")
print(user_stats.describe().apply(lambda s: s.apply('{0:.2f}'.format)))

#### Content popularity trends 

In [ ]:
movie_stats = df.groupby('Movie_ID')['Rating'].agg(['count', 'mean']).reset_index()
movie_stats = pd.merge(movie_stats, movies_df, on='Movie_ID', how='inner')

plt.figure(figsize=(12, 5))
df.groupby(df['Date'].dt.to_period("M")).size().plot(color='teal')
plt.title('Number of Ratings Given Over Time (Monthly)')
plt.xlabel('Date')
plt.ylabel('Number of Ratings')
plt.show()

print("\n--- Top 10 Most Rated Movies ---")
display(movie_stats.sort_values(by='count', ascending=False)[['Title', 'Year', 'count', 'mean']].head(10))

print("\n--- Top 10 Highest Rated Movies (Min 5000 ratings) ---")
popular_movies = movie_stats[movie_stats['count'] > 5000]
display(popular_movies.sort_values(by='mean', ascending=False)[['Title', 'Year', 'count', 'mean']].head(10))

#### Rating Distributions 

In [ ]:
plt.figure(figsize=(8, 5))
rating_counts = df['Rating'].value_counts().sort_index()
ax = sns.barplot(x=rating_counts.index, y=rating_counts.values, palette="viridis")

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))

plt.title('Overall Distribution of Ratings')
plt.xlabel('Rating')
plt.ylabel('Number of Ratings')
plt.show()

print("Rating Percentages:")
print((rating_counts / len(df) * 100).round(2).astype(str) + '%')

#### Data Sparsity Characteristics

In [ ]:
n_users = df['User_ID'].nunique()
n_movies = df['Movie_ID'].nunique()
n_ratings = len(df)

total_possible_interactions = n_users * n_movies

sparsity = 1.0 - (n_ratings / total_possible_interactions)

print(f"Data Sparsity Metrics:")
print(f"Total Users: {n_users:,}")
print(f"Total Movies: {n_movies:,}")
print(f"Total Actual Ratings: {n_ratings:,}")
print(f"Total Possible Matrix Cells: {total_possible_interactions:,}")
print("-" * 30)
print(f"Matrix Sparsity: {sparsity * 100:.4f}%")

In [ ]:
!pip install scikit-surprise

### Loading the saved dataset

In [8]:
print("Loading pre-processed dataset ")
start_time = time.time()

df = pd.read_parquet('netflix_full_processed', engine='pyarrow')

print(f"Loaded {len(df):,} rows in {time.time() - start_time:.2f} seconds!")
print("\nData Schema Verification:")
print(df.dtypes)

Loading pre-processed dataset 
Loaded 100,396,376 rows in 2.37 seconds!

Data Schema Verification:
Movie_ID             int32
User_ID              int32
Rating                int8
Date        datetime64[ns]
dtype: object


### Model Training 

In [ ]:
reader = Reader(rating_scale=(1, 5))

print("Loading 100M rows into Surprise format...")
data = Dataset.load_from_df(df[['User_ID', 'Movie_ID', 'Rating']], reader)

print("Building the full production training set...")
trainset = data.build_full_trainset()

print("Initializing the SVD model...")
algo = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, verbose=True)

start_time = time.time()

algo.fit(trainset)

print(f"Global Training completed in {(time.time() - start_time) / 60:.2f} minutes.")

#### Predicting Unseen Ratings

In [4]:
_, algo = dump.load('svd_netflix_model.pkl')

target_user = 30878
unseen_movie = 12493 

prediction = algo.predict(uid=target_user, iid=unseen_movie)

print(f"User ID: {prediction.uid}")
print(f"Movie ID: {prediction.iid}")
print(f"Estimated Rating: {prediction.est:.2f} Stars")

User ID: 30878
Movie ID: 12493
Estimated Rating: 3.75 Stars


### Recommendation systems 

#### Top k recommendations

In [2]:
print("Loading saved model and datasets...")
_, algo = dump.load('svd_netflix_model.pkl')

ratings_df = pd.read_parquet('netflix_full_processed', engine='pyarrow')

movies_df = pd.read_csv('movie_titles.csv', encoding='ISO-8859-1', 
                        header=None, names=['Movie_ID', 'Year', 'Title'], 
                        on_bad_lines='skip')
movies_df['Year'] = movies_df['Year'].fillna(0).astype(int)

def get_top_k_recommendations(user_id, k=10):
    print(f"\nGenerating Top-{k} Recommendations for User {user_id}...")
    start_time = time.time()
    
    all_movie_ids = set(ratings_df['Movie_ID'].unique())
    user_history = ratings_df[ratings_df['User_ID'] == user_id]
    rated_movie_ids = set(user_history['Movie_ID'])
    
    unrated_movie_ids = list(all_movie_ids - rated_movie_ids)
    
    if not unrated_movie_ids:
        return "User has rated every movie in the database!"

    predictions = []
    for movie_id in unrated_movie_ids:
        pred = algo.predict(uid=user_id, iid=movie_id)
        predictions.append((movie_id, pred.est))
        
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    top_k_predictions = predictions[:k]
    
    recommendations = []
    for movie_id, estimated_rating in top_k_predictions:
        movie_info = movies_df[movies_df['Movie_ID'] == movie_id].iloc[0]
        recommendations.append({
            'Movie_ID': movie_id,
            'Title': movie_info['Title'],
            'Year': movie_info['Year'],
            'Predicted_Rating': round(estimated_rating, 3)
        })
        
    print(f"Completed in {time.time() - start_time:.2f} seconds.")
    return pd.DataFrame(recommendations)

target_user = 30878 
top_10_df = get_top_k_recommendations(user_id=target_user, k=10)
display(top_10_df)

Loading saved model and datasets...

Generating Top-10 Recommendations for User 30878...
Completed in 0.47 seconds.


,Movie_ID,Title,Year,Predicted_Rating
0,4427,The West Wing: Season 3,2001,5.000
1,5519,The West Wing: Season 1,1999,5.000
2,10644,The West Wing: Season 4,2002,5.000
3,13556,The West Wing: Season 2,1999,5.000
4,7569,Dead Like Me: Season 2,2004,4.964
5,7430,Six Feet Under: Season 1,2001,4.956
6,15784,Dead Like Me: Season 1,2003,4.945
7,10080,Six Feet Under: Season 2,2001,4.891
8,3456,Lost: Season 1,2004,4.861
9,5700,Law & Order: Season 14,2003,4.856


#### Sample Recommendations & Recommendation Quality

In [11]:
def analyze_user_quality(user_id, ratings_df, movies_df, algo, k=5):
    print(f"=== Quality Analysis for User {user_id} ===")
    
    user_history = ratings_df[ratings_df['User_ID'] == user_id]
    top_history = user_history[user_history['Rating'] == 5].head(k)
    top_history = pd.merge(top_history, movies_df, on='Movie_ID', how='inner')
    
    print("\n[Historical Favorites (Rated 5 Stars)]:")
    if top_history.empty:
        print("No 5-star movies found for this user.")
    else:
        display(top_history[['Title', 'Year', 'Rating']])

    all_movies = set(ratings_df['Movie_ID'].unique())
    watched_movies = set(user_history['Movie_ID'])
    unseen_movies = list(all_movies - watched_movies)
    
    predictions = [algo.predict(user_id, movie_id) for movie_id in unseen_movies]
    predictions.sort(key=lambda x: x.est, reverse=True)
    
    recommendations = []
    for pred in predictions[:k]:
        title = movies_df[movies_df['Movie_ID'] == pred.iid]['Title'].values[0]
        year = movies_df[movies_df['Movie_ID'] == pred.iid]['Year'].values[0]
        recommendations.append({"Title": title, "Year": year, "Predicted_Rating": round(pred.est, 2)})
        
    print(f"\n[Top {k} Recommendations]:")
    display(pd.DataFrame(recommendations))

analyze_user_quality(user_id=30878, ratings_df=df, movies_df=movies_df, algo=algo, k=5)

=== Quality Analysis for User 30878 ===

[Historical Favorites (Rated 5 Stars)]:


,Title,Year,Rating
0,Spitfire Grill,1996.0,5
1,Gross Anatomy,1989.0,5
2,Dogma,1999.0,5
3,Transformers: Season 3: Part 1,1986.0,5
4,Mississippi Burning,1988.0,5



[Top 5 Recommendations]:


,Title,Year,Predicted_Rating
0,The West Wing: Season 3,2001.0,5.00
1,The West Wing: Season 1,1999.0,5.00
2,The West Wing: Season 4,2002.0,5.00
3,The West Wing: Season 2,1999.0,5.00
4,Dead Like Me: Season 2,2004.0,4.96


#### Demonstrating Success cases

In [13]:
def find_success_case(ratings_df):
    labyrinth_fans = set(ratings_df[(ratings_df['Movie_ID'] == 2802) & (ratings_df['Rating'] == 5)]['User_ID'])
    crystal_fans = set(ratings_df[(ratings_df['Movie_ID'] == 2102) & (ratings_df['Rating'] == 5)]['User_ID'])
    
    niche_users = list(labyrinth_fans.intersection(crystal_fans))
    
    if niche_users:
        print(f"Found a great niche test user: {niche_users[0]}")
        return niche_users[0]
    return None

niche_user_id = find_success_case(df)

if niche_user_id:
    analyze_user_quality(user_id=niche_user_id, ratings_df=df, movies_df=movies_df, algo=algo, k=5)

Found a great niche test user: 1490538
=== Quality Analysis for User 1490538 ===

[Historical Favorites (Rated 5 Stars)]:


,Title,Year,Rating
0,What the #$*! Do We Know!?,2004.0,5
1,Lilo and Stitch,2002.0,5
2,Something's Gotta Give,2003.0,5
3,Spitfire Grill,1996.0,5
4,Dragonheart,1996.0,5



[Top 5 Recommendations]:


,Title,Year,Predicted_Rating
0,Lord of the Rings: The Return of the King: Ext...,2003.0,5
1,Agatha Christie's Poirot: Sad Cypress,2003.0,5
2,The Thin Man Goes Home,1945.0,5
3,Dangerous Minds,1995.0,5
4,Read Or Die,2003.0,5


#### Demonstrating Failure Case

In [14]:
def demonstrate_failure_case(ratings_df, movies_df, algo):
    movie_stats = ratings_df.groupby('Movie_ID')['Rating'].agg(['mean', 'count'])
    
    blockbusters = movie_stats[(movie_stats['count'] > 10000) & (movie_stats['mean'] > 4.5)]
    
    print("=== Failure Case Analysis: The Popularity Bias ===")
    print("Notice how these blockbuster movies often dominate the top recommendations for almost ANY user")
    print("simply because their global bias (b_i) is so high, regardless of the user's actual taste:\n")
    
    top_blockbusters = pd.merge(blockbusters, movies_df, on='Movie_ID', how='inner')
    display(top_blockbusters.sort_values(by='mean', ascending=False)[['Title', 'mean', 'count']].head(5))

demonstrate_failure_case(df, movies_df, algo)

=== Failure Case Analysis: The Popularity Bias ===
Notice how these blockbuster movies often dominate the top recommendations for almost ANY user
simply because their global bias (b_i) is so high, regardless of the user's actual taste:



,Title,mean,count
11,Lord of the Rings: The Return of the King: Ext...,4.723445,73302
4,The Lord of the Rings: The Fellowship of the R...,4.716644,73406
3,Lord of the Rings: The Two Towers: Extended Ed...,4.702672,74897
10,The Shawshank Redemption: Special Edition,4.593358,139624
9,Lord of the Rings: The Return of the King,4.545320,134224


## Evaluation metrics

In [2]:
print("Loading data subset for Evaluation Run...")
raw_data = []
with open('combined_data_1.txt', 'r') as f:
    current_movie = None
    for line in f:
        line = line.strip()
        if line.endswith(':'):
            current_movie = int(line[:-1])
        else:
            user, rating, _ = line.split(',')
            raw_data.append([current_movie, int(user), int(rating)])

df_eval = pd.DataFrame(raw_data, columns=['Movie_ID', 'User_ID', 'Rating'])
del raw_data

Loading data subset for Evaluation Run...


In [4]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_eval[['User_ID', 'Movie_ID', 'Rating']], reader)

print("Splitting data 80/20...")
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print("Training evaluation model...")
algo = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, verbose=True)
algo.fit(trainset)

Splitting data 80/20...
Training evaluation model...
Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19


In [5]:
print("Generating predictions on the test set...")
predictions = algo.test(testset)

Generating predictions on the test set...


### RMSE score

In [10]:
print("\n--- Evaluation Results ---")
rmse = accuracy.rmse(predictions, verbose=False)
print(f" RMSE: {rmse:.4f}")


--- Evaluation Results ---
 RMSE: 0.9003


### MAP@10

In [11]:
def calculate_map_at_k(predictions, k=10, threshold=4.0):
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    average_precisions = []

    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)

        if n_rel == 0:
            continue 
        top_k = user_ratings[:k]

        hits = 0
        sum_precisions = 0

        for i, (est, true_r) in enumerate(top_k):
            if true_r >= threshold:
                hits += 1
                sum_precisions += hits / (i + 1.0)

        ap = sum_precisions / min(n_rel, k)
        average_precisions.append(ap)

    if not average_precisions:
        return 0.0
    return sum(average_precisions) / len(average_precisions)

map_10 = calculate_map_at_k(predictions, k=10, threshold=4.0)
print(f" MAP@10: {map_10:.4f}")

 MAP@10: 0.8177


### Model Comparison

In [13]:
print("Loading combined_data_1.txt for comparative evaluation...")
raw_data = []
with open('combined_data_1.txt', 'r') as f:
    current_movie = None
    for line in f:
        line = line.strip()
        if line.endswith(':'):
            current_movie = int(line[:-1])
        else:
            user, rating, _ = line.split(',')
            raw_data.append([current_movie, int(user), int(rating)])

df_comp = pd.DataFrame(raw_data, columns=['Movie_ID', 'User_ID', 'Rating'])
del raw_data

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_comp[['User_ID', 'Movie_ID', 'Rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

def calculate_map_10(predictions, threshold=4.0):
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))
    average_precisions = []
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        if n_rel == 0: continue 
        top_k = user_ratings[:10]
        hits = 0
        sum_precisions = 0
        for i, (est, true_r) in enumerate(top_k):
            if true_r >= threshold:
                hits += 1
                sum_precisions += hits / (i + 1.0)
        average_precisions.append(sum_precisions / min(n_rel, 10))
    return sum(average_precisions) / len(average_precisions) if average_precisions else 0.0

print("\nTraining Model A: Funk SVD...")
model_a = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02)

start_time = time.time()
model_a.fit(trainset)
svd_train_time = time.time() - start_time

start_time = time.time()
preds_a = model_a.test(testset)
svd_inf_time = time.time() - start_time

svd_rmse = accuracy.rmse(preds_a, verbose=False)
svd_map = calculate_map_10(preds_a)

print("Training Model B: Item-Based Collaborative Filtering...")
sim_options = {'name': 'pearson_baseline', 'user_based': False, 'min_support': 5}
model_b = KNNBasic(sim_options=sim_options, verbose=False)

start_time = time.time()
model_b.fit(trainset)
knn_train_time = time.time() - start_time

start_time = time.time()
preds_b = model_b.test(testset)
knn_inf_time = time.time() - start_time

knn_rmse = accuracy.rmse(preds_b, verbose=False)
knn_map = calculate_map_10(preds_b)

comparison_data = {
    "Metric / Feature": ["Recommendation Quality (RMSE ↓)", "Recommendation Ranking (MAP@10 ↑)", "Training Time Complexity", "Inference Speed (Computational Efficiency)"],
    "Approach 1: Funk SVD (Matrix Factorization)": [f"{svd_rmse:.4f}", f"{svd_map:.4f}", f"{svd_train_time:.2f} seconds", f"{svd_inf_time:.2f} seconds"],
    "Approach 2: KNN (Item-Based CF)": [f"{knn_rmse:.4f}", f"{knn_map:.4f}", f"{knn_train_time:.2f} seconds", f"{knn_inf_time:.2f} seconds"]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n=== SECTION C: APPROACH COMPARISON ===")
display(comparison_df)

Loading combined_data_1.txt for comparative evaluation...

Training Model A: Funk SVD...
Training Model B: Item-Based Collaborative Filtering...

=== SECTION C: APPROACH COMPARISON ===


,Metric / Feature,Approach 1: Funk SVD (Matrix Factorization),Approach 2: KNN (Item-Based CF)
0,Recommendation Quality (RMSE ↓),0.9000,0.9266
1,Recommendation Ranking (MAP@10 ↑),0.8177,0.8002
2,Training Time Complexity,154.93 seconds,152.60 seconds
3,Inference Speed (Computational Efficiency),10.72 seconds,894.18 seconds
